## 1. Initialize Project Environment
Import libraries for correlation computation, graph construction, and community detection.

In [ ]:
from __future__ import annotations

import logging
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Dict

import numpy as np
import pandas as pd
import networkx as nx

logging.basicConfig(level=logging.INFO, format="[%(levelname)s] %(message)s")


def locate_repo_root() -> Path:
    """Find the repository root by looking for data folder."""
    here = Path().resolve()
    for base in [here, *here.parents]:
        if (base / "data").exists():
            return base
    raise FileNotFoundError("Could not locate repository root")


REPO_ROOT = locate_repo_root()
ARTIFACTS = REPO_ROOT / "labs/07_network_viz/assignments/artifacts"
ARTIFACTS.mkdir(parents=True, exist_ok=True)

logging.info("Repo root: %s", REPO_ROOT)
logging.info("Artifacts directory: %s", ARTIFACTS)

## 2. Define Configuration Parameters
Centralize correlation method, threshold settings, and output paths.

In [ ]:
@dataclass
class NetworkConfig:
    handle: str
    corr_method: str = "spearman"  # "pearson" or "spearman"
    use_abs_corr: bool = True
    adj_threshold: float = 0.85
    weighted: bool = False
    export_dir: Path = None

    def __post_init__(self):
        if self.export_dir is None:
            self.export_dir = ARTIFACTS

    def describe(self) -> Dict[str, str]:
        info = asdict(self)
        info["export_dir"] = str(info["export_dir"])
        return info


CONFIG = NetworkConfig(
    handle="AndreiCod",
    corr_method="spearman",
    use_abs_corr=True,
    adj_threshold=0.85,
)
CONFIG.describe()

In [ ]:
# Load preprocessed data from Task 1
preprocessed_path = ARTIFACTS / "task1_preprocessed_expression.csv"
if not preprocessed_path.exists():
    # Fallback to original Lab 6 data
    logging.warning("Task 1 output not found, loading from Lab 6 data...")
    preprocessed_path = REPO_ROOT / "data/work/AndreiCod/lab06/expression_matrix.csv"

expr = pd.read_csv(preprocessed_path, index_col=0)
logging.info("Loaded expression data: %d genes × %d samples", expr.shape[0], expr.shape[1])
expr.head()

## 3. Implement Core Functionality
Compute correlation matrix, build adjacency matrix, construct graph, and detect modules.

In [ ]:
def compute_correlation_matrix(df: pd.DataFrame, method: str = "spearman") -> pd.DataFrame:
    """Compute gene-gene correlation matrix.

    Args:
        df: Expression matrix (genes as rows, samples as columns)
        method: 'pearson' or 'spearman'

    Returns:
        Correlation matrix (genes × genes)
    """
    # Transpose so genes become columns for correlation
    corr = df.T.corr(method=method)
    logging.info("Computed %s correlation matrix: %d × %d", method, corr.shape[0], corr.shape[1])
    return corr


def build_adjacency_matrix(
    corr: pd.DataFrame,
    threshold: float,
    use_abs: bool = True,
    weighted: bool = False
) -> pd.DataFrame:
    """Build adjacency matrix from correlation matrix.

    Args:
        corr: Correlation matrix
        threshold: Minimum correlation for an edge
        use_abs: Whether to use absolute correlation values
        weighted: If True, keep correlation values; if False, binary adjacency

    Returns:
        Adjacency matrix
    """
    if use_abs:
        corr_vals = corr.abs()
    else:
        corr_vals = corr

    if weighted:
        adj = corr_vals.where(corr_vals >= threshold, 0)
    else:
        adj = (corr_vals >= threshold).astype(int)

    # Remove self-loops
    np.fill_diagonal(adj.values, 0)

    n_edges = (adj.values > 0).sum() // 2  # undirected
    logging.info("Adjacency matrix: threshold=%.2f, edges=%d", threshold, n_edges)
    return adj


def build_graph(adj: pd.DataFrame) -> nx.Graph:
    """Build NetworkX graph from adjacency matrix."""
    G = nx.from_pandas_adjacency(adj)

    # Remove isolated nodes
    isolates = list(nx.isolates(G))
    G.remove_nodes_from(isolates)

    logging.info("Graph: %d nodes, %d edges (removed %d isolates)",
                 G.number_of_nodes(), G.number_of_edges(), len(isolates))
    return G

In [ ]:
# Step 1: Compute correlation matrix
corr_matrix = compute_correlation_matrix(expr, CONFIG.corr_method)
corr_matrix.head()

In [ ]:
# Step 2: Build adjacency matrix
adj_matrix = build_adjacency_matrix(
    corr_matrix,
    threshold=CONFIG.adj_threshold,
    use_abs=CONFIG.use_abs_corr,
    weighted=CONFIG.weighted
)
adj_matrix.head()

In [ ]:
# Step 3: Build graph
G = build_graph(adj_matrix)
print(f"Network summary:")
print(f"  Nodes: {G.number_of_nodes()}")
print(f"  Edges: {G.number_of_edges()}")
print(f"  Density: {nx.density(G):.4f}")
print(f"  Connected components: {nx.number_connected_components(G)}")

In [ ]:
def detect_modules_louvain(G: nx.Graph) -> Dict[str, int]:
    """Detect modules using Louvain community detection algorithm.

    Returns:
        Dict mapping gene names to module IDs
    """
    try:
        # Try networkx-community louvain
        from networkx.algorithms.community import louvain_communities
        communities = louvain_communities(G, seed=42)
        gene2module = {}
        for module_id, community in enumerate(communities):
            for gene in community:
                gene2module[gene] = module_id
        logging.info("Louvain detected %d modules", len(communities))
    except ImportError:
        # Fallback to greedy modularity
        from networkx.algorithms.community import greedy_modularity_communities
        communities = greedy_modularity_communities(G)
        gene2module = {}
        for module_id, community in enumerate(communities):
            for gene in community:
                gene2module[gene] = module_id
        logging.info("Greedy modularity detected %d modules", len(communities))

    return gene2module


# Detect modules
gene2module = detect_modules_louvain(G)
print(f"\nModule distribution:")
module_counts = pd.Series(gene2module).value_counts().sort_index()
for mod, count in module_counts.items():
    print(f"  Module {mod}: {count} genes")

## 4. Validate with Unit Tests
Ensure correlation and adjacency functions work correctly.

In [ ]:
def test_correlation_matrix():
    """Test correlation matrix computation."""
    test_df = pd.DataFrame(
        {"S1": [1, 2, 3], "S2": [2, 4, 6], "S3": [3, 6, 9]},
        index=["G1", "G2", "G3"]
    )
    corr = compute_correlation_matrix(test_df, "pearson")
    # Perfect correlation for linearly related data
    assert np.isclose(corr.loc["G1", "G2"], 1.0)
    assert corr.shape == (3, 3)


def test_adjacency_matrix():
    """Test adjacency matrix building."""
    test_corr = pd.DataFrame(
        [[1.0, 0.9, 0.3], [0.9, 1.0, 0.5], [0.3, 0.5, 1.0]],
        index=["A", "B", "C"],
        columns=["A", "B", "C"]
    )
    adj = build_adjacency_matrix(test_corr, threshold=0.8, use_abs=True, weighted=False)
    assert adj.loc["A", "B"] == 1  # 0.9 >= 0.8
    assert adj.loc["A", "C"] == 0  # 0.3 < 0.8
    assert adj.loc["A", "A"] == 0  # No self-loops


test_correlation_matrix()
test_adjacency_matrix()
logging.info("All network construction tests passed.")

## 5. Export Results
Save the module mapping and network statistics.

In [ ]:
# Export modules CSV (required deliverable)
modules_df = pd.DataFrame([
    {"Gene": gene, "Module": module}
    for gene, module in sorted(gene2module.items())
])
modules_path = CONFIG.export_dir / f"modules_tp53_{CONFIG.handle}.csv"
modules_df.to_csv(modules_path, index=False)
logging.info("[OK] Module mapping saved to: %s", modules_path.resolve())

# Export correlation matrix for reference
corr_path = CONFIG.export_dir / "task2_correlation_matrix.csv"
corr_matrix.to_csv(corr_path)
logging.info("[OK] Correlation matrix saved to: %s", corr_path.resolve())

# Export adjacency matrix for reference
adj_path = CONFIG.export_dir / "task2_adjacency_matrix.csv"
adj_matrix.to_csv(adj_path)
logging.info("[OK] Adjacency matrix saved to: %s", adj_path.resolve())

# Export network statistics
stats = {
    "nodes": G.number_of_nodes(),
    "edges": G.number_of_edges(),
    "density": nx.density(G),
    "num_modules": len(set(gene2module.values())),
    "corr_method": CONFIG.corr_method,
    "threshold": CONFIG.adj_threshold,
}
stats_df = pd.DataFrame([stats])
stats_path = CONFIG.export_dir / "task2_network_stats.csv"
stats_df.to_csv(stats_path, index=False)
logging.info("[OK] Network statistics saved to: %s", stats_path.resolve())

print(f"\n✓ Task 2 complete: {len(set(gene2module.values()))} modules detected.")